In [ ]:
#%% 必要なライブラリをインポート
import pandas as pd
import numpy as np
from tslearn.clustering import TimeSeriesKMeans, KShape
from tslearn.metrics import dtw
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
import matplotlib.pyplot as plt
import os
import datetime

#%% 前処理の改良
def preprocess_demand_data(input_folder, months=[6, 12]):
    list_demand = []
    list_part_demand = []

    data_range = pd.date_range(start=datetime.datetime(2012, 1, 1), end=datetime.datetime(2016, 1, 1), freq="h")
    for file in os.listdir(input_folder):
        df = pd.read_csv(os.path.join(input_folder, file), index_col=0, parse_dates=True)
        df = df.loc[~df.index.duplicated(), :]  # 重複したインデックスを削除
        if "Demand_Power" not in df.columns or df["Demand_Power"].max() > df["Demand_Power"].mean() * 5:
            continue

        # 1週間分のデータを抽出
        list_series = []
        for month in months:
            one_week_data = extract_1week(df, month)
            if one_week_data is not None:
                list_series.append(one_week_data)

        if list_series:
            list_part_demand.append(pd.concat(list_series))
        list_demand.append(df["Demand_Power"].reset_index(drop=True))

    df_demand = pd.concat(list_demand, axis=1).T
    df_demand = df_demand.ffill().bfill()  # 欠損値の補完
    return df_demand, list_part_demand

def extract_1week(df, month):
    try:
        # 対象月の火曜日のデータを抽出
        first_week = df[df.index.month == month][df.index.weekday == 1][:24 * 7]
        return first_week["Demand_Power"] / first_week["Demand_Power"].max()  # 正規化
    except Exception as e:
        return None

#%% データの読み込みと前処理
input_folder = "BuildingDemand_Input"
df_demand, list_part_demand = preprocess_demand_data(input_folder)

# 正規化と時系列データの整形
nm_demanddata = TimeSeriesScalerMeanVariance().fit_transform(df_demand.values)

#%% クラスタリングの実施（k-meansとk-shapeの比較）
n_clusters = 3
models = {
    "k-means": TimeSeriesKMeans(n_clusters=n_clusters, metric="dtw", verbose=True, random_state=42),
    "k-shape": KShape(n_clusters=n_clusters, verbose=True, random_state=42),
}

results = {}
for model_name, model in models.items():
    print(f"Running {model_name} clustering...")
    y_pred = model.fit_predict(nm_demanddata)
    results[model_name] = (model, y_pred)

#%% クラスタリング結果の可視化
for model_name, (model, y_pred) in results.items():
    print(f"\nResults for {model_name} clustering:")
    for cluster_idx in range(n_clusters):
        fig1, ax1 = plt.subplots()
        fig2, ax2 = plt.subplots()
        cluster_samples = [nm_demanddata[idx] for idx in range(len(y_pred)) if y_pred[idx] == cluster_idx]
        errors = [
            np.sum(np.abs(model.cluster_centers_[cluster_idx].ravel() - sample.ravel()))
            for sample in cluster_samples
        ]

        # 上位10サンプルをプロット
        top_samples = sorted(zip(cluster_samples, errors), key=lambda x: x[1])[:10]
        for sample, _ in top_samples:
            ax2.plot(sample.ravel(), "k-", alpha=0.3)

        # クラスタ中心と全体プロット
        for sample in cluster_samples:
            ax1.plot(sample.ravel(), "k-", alpha=0.1)
        ax1.plot(model.cluster_centers_[cluster_idx].ravel(), "r-", label="Cluster Center")
        ax1.legend()

        # ラベルと保存
        ax1.set_title(f"Cluster {cluster_idx} ({model_name})")
        ax1.set_xlabel("Time Index")
        ax1.set_ylabel("Normalized Demand")
        plt.savefig(f"IMG/{model_name}_cluster_{cluster_idx}.png")
        plt.show()

        print(f"Cluster {cluster_idx}: {len(cluster_samples)} samples, Mean Error: {np.mean(errors):.2f}")
